[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/Multimodal-Deep-Learning/blob/main/07_traing_problem_and%20soultion/03_overfitting_underfitting/03_overfitting_underfitting.ipynb)

# 03. Overfitting, Underfitting & Regularization

---


In [ ]:
# ============================================================
#  Colab Setup (run this cell first if on Google Colab)
# ============================================================
import os, sys

if 'google.colab' in sys.modules:
    !git clone https://github.com/Gaurav14cs17/Multimodal-Deep-Learning.git
    os.chdir('Multimodal-Deep-Learning')
    os.chdir('07_traing_problem_and soultion/03_overfitting_underfitting')
    !pip install -q torch torchvision matplotlib numpy
else:
    nb_dir = os.getcwd()
    if not os.path.basename(nb_dir) == '03_overfitting_underfitting':
        os.chdir(os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', '..', '03_overfitting_underfitting'))
    sys.path.append(os.path.join(os.getcwd(), '..', '..'))

print(f'Working directory: {os.getcwd()}')


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
plt.rcParams.update({'figure.figsize': (10, 5), 'font.size': 11})


## 1. Diagnosing — Four Scenarios

```
Scenario 1: Good fit     train ↓  val ↓  (both decrease)
Scenario 2: Overfit        train ↓  val ↑  (gap widens)
Scenario 3: Underfit       train ↑  val ↑  (both high)
Scenario 4: Double descent val has second dip after interpolation
```


## 2. Bias-Variance Decomposition

$$
\mathbb{E}[(y - \hat{f})^2] = \text{Bias}^2 + \text{Variance} + \text{Noise}
$$

Overfitting = high variance. Underfitting = high bias.


## 3. Dropout — Expected Value Preservation

$$
\tilde{h} = h \odot m / (1-p), \quad m \sim \text{Bernoulli}(1-p)
$$

$E[\tilde{h}] = h \cdot (1-p) \cdot \frac{1}{1-p} = h$ — inference uses all units scaled by $(1-p)$.


In [ ]:
p = 0.5
h = torch.ones(1000, 64)
drops = [F.dropout(h, p=p, training=True).mean().item() for _ in range(200)]
print(f'E[h]={h.mean():.3f}, E[dropout(h)]={np.mean(drops):.3f} (preserved)')


## 4. Weight Decay / L2

$$
\mathcal{L}_{reg} = \mathcal{L} + \frac{\lambda}{2}\lVert W \rVert^2
$$

Gradient update: $W \leftarrow (1 - \eta\lambda)W - \eta\nabla\mathcal{L}$ — explicit shrinkage each step.


## 5. Label Smoothing

$$
y_{smooth} = (1-\epsilon) y + \epsilon / K
$$

Prevents overconfident logits; softens cross-entropy targets.


## 6. Mixup

$$
\tilde{x} = \lambda x_i + (1-\lambda)x_j, \quad \lambda \sim \text{Beta}(\alpha, \alpha)
$$

Linearly interpolates inputs and labels → smoother decision boundaries (Zhang et al. 2018).


In [ ]:
class OverfitNet(nn.Module):
    def __init__(self, hidden=128, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, hidden), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 2))
    def forward(self, x): return self.net(x)

def make_data(n=30):
    X = torch.randn(n, 2); y = (X[:, 0] + X[:, 1] > 0).long()
    return X, y

def train_model(dropout=0.0, wd=0.0, steps=300):
    X, y = make_data()
    model = OverfitNet(dropout=dropout)
    opt = torch.optim.Adam(model.parameters(), lr=1e-2, weight_decay=wd)
    train_loss, val_loss = [], []
    for step in range(steps):
        model.train(); opt.zero_grad()
        loss = F.cross_entropy(model(X), y)
        loss.backward(); opt.step()
        train_loss.append(loss.item())
        model.eval()
        with torch.no_grad():
            vl = F.cross_entropy(model(X + 0.5*torch.randn_like(X)), y).item()
        val_loss.append(vl)
    return train_loss, val_loss

t0, v0 = train_model()
t1, v1 = train_model(dropout=0.3, wd=0.1)
fig, ax = plt.subplots(1, 2, figsize=(12,4))
ax[0].plot(t0, label='train'); ax[0].plot(v0, label='val'); ax[0].set_title('No regularization'); ax[0].legend()
ax[1].plot(t1, label='train'); ax[1].plot(v1, label='val'); ax[1].set_title('Dropout + weight decay'); ax[1].legend()
plt.show()


## 7. Double Descent

As model capacity crosses the **interpolation threshold** (can fit training data exactly), test error can **decrease again** — contradicting classical U-shaped bias-variance curve (Nakkiran et al. 2019).


## 8. Early Stopping — Implicit Regularization

Stop when validation loss increases for $P$ consecutive epochs. Equivalent to limiting effective model capacity — prevents overfitting without explicit penalty.


In [ ]:
# Label smoothing demo
K, eps = 10, 0.1
y_hard = torch.zeros(K); y_hard[3] = 1.0
y_smooth = (1 - eps) * y_hard + eps / K
print('Hard label:', y_hard.tolist())
print('Smoothed: ', [round(v, 3) for v in y_smooth.tolist()])


## 9. Solution Decision Table

| Problem | Solution | When |
|---------|----------|------|
| Overfit | Dropout, WD, augmentation | val > train |
| Underfit | More layers, less reg | both losses high |
| Overconfident | Label smoothing | calibration matters |
| Small data | Mixup, augmentation | limited samples |


## References & Further Reading

- Srivastava et al. (2014) — Dropout — [JMLR](https://jmlr.org/papers/v15/srivastava14a.html)
- Zhang et al. (2018) — mixup — [arXiv:1710.09412](https://arxiv.org/abs/1710.09412)
- Nakkiran et al. (2019) — Deep Double Descent — [arXiv:1912.02292](https://arxiv.org/abs/1912.02292)
